In [1]:
import os
from PIL import Image

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from sklearn.model_selection import train_test_split
import json
import sys

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Conv2D, GlobalAveragePooling2D, Flatten, Dense, Activation, Dropout, BatchNormalization
from tensorflow.keras import regularizers

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.applications import  EfficientNetV2B2

2025-12-22 07:41:07.411588: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766389267.630526      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766389267.689907      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
data_dir = '/kaggle/input/indian-birds/Birds_25'


filepaths = []
labels = []

for split_folder in ['train', 'valid']:
    split_path = os.path.join(data_dir, split_folder)
  
    if os.path.isdir(split_path):
        for class_label in os.listdir(split_path):
            class_folder_path = os.path.join(split_path, class_label)
        
            if os.path.isdir(class_folder_path):
                for file_name in os.listdir(class_folder_path):
                    file_path = os.path.join(class_folder_path, file_name)
           
                    if os.path.isfile(file_path):
                        filepaths.append(file_path)
                        labels.append(class_label)

In [3]:
df = pd.DataFrame({
    'filepaths': filepaths,
    'labels': labels,
})

In [4]:
df  # view data

,filepaths,labels
0,/kaggle/input/indian-birds/Birds_25/train/Comm...,Common-Rosefinch
1,/kaggle/input/indian-birds/Birds_25/train/Comm...,Common-Rosefinch
2,/kaggle/input/indian-birds/Birds_25/train/Comm...,Common-Rosefinch
3,/kaggle/input/indian-birds/Birds_25/train/Comm...,Common-Rosefinch
4,/kaggle/input/indian-birds/Birds_25/train/Comm...,Common-Rosefinch
...,...,...
37495,/kaggle/input/indian-birds/Birds_25/valid/Nort...,Northern-Lapwing
37496,/kaggle/input/indian-birds/Birds_25/valid/Nort...,Northern-Lapwing
37497,/kaggle/input/indian-birds/Birds_25/valid/Nort...,Northern-Lapwing
37498,/kaggle/input/indian-birds/Birds_25/valid/Nort...,Northern-Lapwing


In [5]:
strat = df['labels']

train_df, dummy_df = train_test_split( df, train_size=0.8, shuffle=True, random_state=42, stratify=strat) 

strat = dummy_df['labels']

valid_df, test_df = train_test_split( dummy_df, train_size=0.5, shuffle=True, random_state=42, stratify=strat)

In [6]:
gen = ImageDataGenerator() # Create image generator without augmentation 

In [7]:
train_gen = gen.flow_from_dataframe(train_df, x_col='filepaths', y_col='labels', target_size=(300, 300), 
                                    class_mode='categorical', color_mode='rgb', shuffle=True, batch_size=16)

valid_gen = gen.flow_from_dataframe(valid_df, x_col='filepaths', y_col='labels', target_size=(300, 300), 
                                    class_mode='categorical', color_mode='rgb', shuffle=True, batch_size=8)

test_gen = gen.flow_from_dataframe(test_df, x_col='filepaths', y_col='labels', target_size=(300, 300), 
                                   class_mode='categorical', color_mode='rgb', shuffle=False, batch_size=8)

Found 30000 validated image filenames belonging to 25 classes.
Found 3750 validated image filenames belonging to 25 classes.
Found 3750 validated image filenames belonging to 25 classes.


In [8]:
base_model = EfficientNetV2B2(
    weights='imagenet',
    include_top=False,
    input_shape=(300, 300, 3)
)

base_model.trainable = False

model = Sequential([
    base_model,
    
    GlobalAveragePooling2D(), 

    Dense(256),  
    BatchNormalization(),  
    Activation('relu'),  
    Dropout(0.3),  

    Dense(128), 
    BatchNormalization(),
    Activation('relu'),  
    Dropout(0.3),

    Dense(25, activation='softmax')  
])

I0000 00:00:1766389439.786509      20 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1766389439.787136      20 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


35839040/35839040 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [9]:
model.compile( optimizer=Adam(learning_rate=0.001), loss = 'categorical_crossentropy', metrics=['accuracy']
)

In [10]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, verbose=1)
]

In [11]:
history = model.fit( 
                    train_gen,  steps_per_epoch=len(train_gen),
                    epochs=10,                      
                    validation_data=valid_gen,  validation_steps=len(valid_gen),
                    verbose=1
                   )

/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10


I0000 00:00:1766389466.577197      61 service.cc:148] XLA service 0x7b230801a6c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1766389466.578204      61 service.cc:156]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1766389466.578226      61 service.cc:156]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1766389469.263042      61 cuda_dnn.cc:529] Loaded cuDNN version 90300


   2/1875 ━━━━━━━━━━━━━━━━━━━━ 2:03 66ms/step - accuracy: 0.0156 - loss: 3.8122       

I0000 00:00:1766389484.555624      61 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1875/1875 ━━━━━━━━━━━━━━━━━━━━ 678s 340ms/step - accuracy: 0.7693 - loss: 0.8634 - val_accuracy: 0.9709 - val_loss: 0.0955
Epoch 2/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 247s 132ms/step - accuracy: 0.9189 - loss: 0.2781 - val_accuracy: 0.9779 - val_loss: 0.0688
Epoch 3/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 231s 123ms/step - accuracy: 0.9327 - loss: 0.2192 - val_accuracy: 0.9819 - val_loss: 0.0575
Epoch 4/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 259s 138ms/step - accuracy: 0.9396 - loss: 0.2008 - val_accuracy: 0.9795 - val_loss: 0.0624
Epoch 5/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 246s 131ms/step - accuracy: 0.9474 - loss: 0.1743 - val_accuracy: 0.9837 - val_loss: 0.0532
Epoch 6/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 266s 142ms/step - accuracy: 0.9494 - loss: 0.1605 - val_accuracy: 0.9856 - val_loss: 0.0457
Epoch 7/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 266s 142ms/step - accuracy: 0.9513 - loss: 0.1516 - val_accuracy: 0.9864 - val_loss: 0.0396
Epoch 8/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 247s 132ms/step - accuracy: 0.9

In [12]:
loss, accuracy = model.evaluate(
    test_gen, steps=len(test_gen),
    verbose=0
)

In [13]:
print(f"Loss : {loss:.4f}")
print(f"Accuracy : {accuracy:.4f}")

Loss : 0.0391
Accuracy : 0.9896


In [14]:
# ...existing code...
# Save artifacts without using os or path module — saves to notebook current working directory


# 1) SavedModel (directory)
try:
    model.save("exported_model")
    print("SavedModel -> exported_model/")
except Exception as e:
    print("SavedModel failed:", e)




SavedModel failed: Invalid filepath extension for saving. Please add either a `.keras` extension for the native Keras format (recommended) or a `.h5` extension. Use `model.export(filepath)` if you want to export a SavedModel for use with TFLite/TFServing/etc. Received: filepath=exported_model.


In [15]:
# 2) Single-file HDF5 full-model
try:
    model.save("exported_model.h5")
    print("Full HDF5 -> exported_model.h5")
except Exception as e:
    print("Full HDF5 save failed:", e)

Full HDF5 -> exported_model.h5


In [16]:
# 3) Weights-only HDF5
try:
    model.save_weights("weights.h5")
    print("Weights -> weights.h5")
except Exception as e:
    print("Weights save failed:", e)


Weights save failed: The filename must end in `.weights.h5`. Received: filepath=weights.h5


In [17]:
# 4) JSON architecture
try:
    with open("model.json", "w", encoding="utf-8") as f:
        f.write(model.to_json())
    print("Model JSON -> model.json")
except Exception as e:
    print("Model JSON save failed:", e)

Model JSON -> model.json


In [18]:
# 5) Training history (if present)
try:
    if 'history' in globals() and hasattr(history, "history"):
        with open("history.json", "w", encoding="utf-8") as f:
            json.dump(history.history, f, indent=2)
        print("History -> history.json")
except Exception as e:
    print("History save failed:", e)

History -> history.json


In [19]:
# 6) Labels / class mapping
try:
    if 'class_names' in globals():
        with open("labels.txt", "w", encoding="utf-8") as f:
            f.write("\n".join(map(str, class_names)))
        print("Labels -> labels.txt (from class_names)")
    elif 'train_gen' in globals() and hasattr(train_gen, "class_indices"):
        inv = {v: k for k, v in train_gen.class_indices.items()}
        with open("class_indices.json", "w", encoding="utf-8") as f:
            json.dump(inv, f, indent=2)
        print("Class indices -> class_indices.json (from train_gen)")
    else:
        nc = int(model.output_shape[-1])
        with open("labels.txt", "w", encoding="utf-8") as f:
            f.write("\n".join([str(i) for i in range(nc)]))
        print("Fallback labels -> labels.txt (0..N-1)")
except Exception as e:
    print("Labels save failed:", e)

Class indices -> class_indices.json (from train_gen)


In [20]:
# 7) Metadata
try:
    meta = {
        "input_shape": list(model.input_shape[1:]) if model.input_shape is not None else None,
        "output_classes": int(model.output_shape[-1]) if model.output_shape else None,
        "tensorflow_version": tf.__version__
    }
    with open("metadata.json", "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)
    print("Metadata -> metadata.json")
except Exception as e:
    print("Metadata save failed:", e)

Metadata -> metadata.json


In [21]:
# 8) requirements (best-effort) — use shell to write pip freeze
try:
    get_ipython().system('pip freeze > requirements.txt')
    print("requirements.txt written")
except Exception as e:
    print("requirements.txt failed:", e)

print("Export complete. Check the notebook working directory for the files.")

requirements.txt written
Export complete. Check the notebook working directory for the files.
